# Match Process2 to Cognospeak files
Apparently, there is no spreadsheet anymore that matches the Process2 to CognoSpeak files. For Process2, participants have been assigned a new participant ID which does not match the CognoSpeak one anymore. This notebook tries to **re-identify the Process2 participants in the wider CognoSpeak dataset** because I need answers to the other questions (not just fluency and CTD).

**Other problems with the data**: some participants are from old and some from new CognoSpeak. The questions don't fully match. However, they are really similar

**approach for matching **
- combine meta data for old and new cognospeak data 
- try to match participants with the metadata present in Process2 and CognoSpeak (age, Moca, gender, diagnosis)
- for files where there is no exact match: try to then either match on WER in transcripts or audio files 
- problems with audio files and transcripts: some of the aufio files have been trimeed; some of the transcript files have been adjusted

## combining meta data files for CognoSpeak new and old 

In [139]:
import pandas as pd

In [140]:
# read in all metadata csv files 

BASE_DIR = "/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/raw/"

meta_process = pd.read_csv(BASE_DIR + "meta-info.csv")
meta_CognoNew = pd.read_csv(BASE_DIR + "CognoSpeak_metadata__2026_06_07.csv")
meta_CognoOld = pd.read_csv(BASE_DIR + "OldCognoSpeak.csv")

In [141]:
# print columns for all dataframes 

print("Columns in meta_process:")
print(meta_process.columns)

print("Columns in meta_CognoNew:")
print(meta_CognoNew.columns)

print("Columns in meta_CognoOld:")
print(meta_CognoOld.columns)

Columns in meta_process:
Index(['IDs', 'diagnosis', 'age', 'gender', 'MMSE', 'Split'], dtype='str')
Columns in meta_CognoNew:
Index(['research_ID', 'pseudo_unq_id', 'assessment_ID', 'dir_name',
       'assess_date', 'age', 'diagnosis', 'diagnosis_type', 'referral',
       'ethnicity', 'gender', 'ENGLISH_FIRST_LANG', 'ENGLISH_ABILITY',
       'FIRST_LANG', 'EDUCATION_INFO', 'REGION', 'OTHER_CONDITIONS', 'Q1',
       'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12',
       'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'PHQ-9', 'GAD-7',
       'EQ-5D', 'REPEAT', 'ASSESS_GAPS_days', 'CONSENT', 'assessment_type',
       'trans_txt', 'MoCA Visuospatial / Executive', 'MoCA Naming',
       'MoCA Attention', 'MoCA Language', 'MoCA Abstraction',
       'MoCA Delayed Recall', 'MoCA Orientation', 'MoCA Total score',
       'MoCA words list', 'MoCA Memory Index Score', 'MoCA years of education',
       'RUDAS Visuospatial Orientation', 'RUDAS Praxis',
       'RUDAS Visuoconstr

In [142]:
meta_process = meta_process[["IDs", "diagnosis", "age", "gender", "MMSE"]]
meta_CognoNew = meta_CognoNew[["dir_name", "diagnosis", "age", "gender", "MMSE"]]
meta_CognoOld = meta_CognoOld[["Unique reference", "Diagnosis", "Age", "Gender", "ConvertedMMSE"]]
meta_CognoOld = meta_CognoOld.rename(columns={"Unique reference": "dir_name", "Diagnosis": "diagnosis", "Age": "age", "Gender": "gender", "ConvertedMMSE": "MMSE"})

In [143]:
print("Columns in meta_process:")
print(meta_process.columns)

print("Columns in meta_CognoNew:")
print(meta_CognoNew.columns)

print("Columns in meta_CognoOld:")
print(meta_CognoOld.columns)

Columns in meta_process:
Index(['IDs', 'diagnosis', 'age', 'gender', 'MMSE'], dtype='str')
Columns in meta_CognoNew:
Index(['dir_name', 'diagnosis', 'age', 'gender', 'MMSE'], dtype='str')
Columns in meta_CognoOld:
Index(['dir_name', 'diagnosis', 'age', 'gender', 'MMSE'], dtype='str')


In [144]:
# Start with CognoSpeak new only; old CognoSpeak matching is trickier and deferred for later.
meta_cogno = meta_CognoNew.copy()

In [145]:
meta_process.head()

,IDs,diagnosis,age,gender,MMSE
0,PROCESS-2_rec__001,HC,79,male,NaN
1,PROCESS-2_rec__002,HC,75,male,NaN
2,PROCESS-2_rec__003,MCI,68,female,28.0
3,PROCESS-2_rec__004,HC,74,male,NaN
4,PROCESS-2_rec__005,HC,79,male,NaN


In [146]:
meta_CognoOld.head()

,dir_name,diagnosis,age,gender,MMSE
0,264,HC,78.0,F,28.0
1,265,HC,72.0,M,29.0
2,266,HC,71.0,M,29.0
3,268,HC,70.0,F,28.0
4,269,HC,69.0,M,29.0


In [147]:
meta_CognoNew['diagnosis'].value_counts()

diagnosis
HC                   2288
Unknown               548
MCI                   285
Dementia              109
Stroke                100
MND                    23
FMD                    20
NC_FMD                  7
FCD                     7
Mild_Vas_Dementia       6
Dementia (FTD)          2
Park_Dementia           2
Name: count, dtype: int64

## Reverse-matching: find Process2 participants inside the wider CognoSpeak set

There is no shared ID, so this matches on demographics/scores instead — **exact** match required on:
- `gender`
- `age`
- `diagnosis` (mapped onto Process2's HC/MCI/Dementia buckets, see normalization below)
- `MMSE`, but only where it's available on both sides — Process2 HCs mostly have no MMSE, so for those rows this simply isn't used as a constraint

This produces, for every Process2 participant, the exact-match candidate set. A participant with exactly one candidate can be taken as matched; a participant with 0 or 2+ candidates needs the transcript/audio WER disambiguation step mentioned in the intro (0 candidates likely means the exact-match constraints are too strict for that person — e.g. a 1-year age drift between the two records).</cell id="placeholder">

In [148]:
import numpy as np

# --- normalize gender vocab: process/CognoNew already use male/female(/undisclosed/other);
# CognoOld uses M/F ---

def normalize_gender(g):
    if pd.isna(g):
        return np.nan
    g = str(g).strip().lower()
    return {"m": "male", "f": "female"}.get(g, g)

meta_process["gender_norm"] = meta_process["gender"].apply(normalize_gender)
meta_cogno["gender_norm"] = meta_cogno["gender"].apply(normalize_gender)

meta_process["gender_norm"].value_counts(dropna=False)

gender_norm
male      202
female    198
Name: count, dtype: int64

In [149]:
# --- normalize diagnosis vocab onto the Process2 buckets (HC / MCI / Dementia) ---
# Process2 already uses these three labels directly.
# CognoNew and CognoOld use different, wider label sets. Only labels we are
# confident belong in one of the three buckets are mapped below; everything
# else is left as NaN (no diagnosis bonus/penalty during matching) rather than
# guessed. CognoOld's SS/ND/DPD/PD/VCI codes in particular are NOT mapped here
# because their meaning isn't obvious from the data alone -> fill these in if
# you know what they stand for, otherwise they'll just be treated as "unknown".

diagnosis_canonical_map = {
    "HC": "HC",
    "MCI": "MCI",
    "Dementia": "Dementia",
    "Mild_Vas_Dementia": "Dementia",
    "Dementia (FTD)": "Dementia",
    "Park_Dementia": "Dementia",
}

meta_cogno["diagnosis_norm"] = meta_cogno["diagnosis"]
meta_process["diagnosis_norm"] = meta_process["diagnosis"]

#meta_process["diagnosis_norm"] = meta_process["diagnosis"].map(diagnosis_canonical_map)
#meta_cogno["diagnosis_norm"] = meta_cogno["diagnosis"].map(diagnosis_canonical_map)

#print("Unmapped diagnosis labels in meta_cogno (left as unknown):")
#print(sorted(meta_cogno.loc[meta_cogno["diagnosis_norm"].isna(), "diagnosis"].dropna().unique()))

In [150]:
# give meta_cogno a single id column to report back, since research_ID is what
# both CognoNew and CognoOld were renamed to earlier
meta_cogno = meta_cogno.reset_index(drop=True)

def get_candidates(process_row, cogno_df):
    cands = cogno_df.copy()

    # exact match on gender, age, diagnosis (normalized)
    cands = cands[cands["gender_norm"] == process_row["gender_norm"]]
    cands = cands[cands["age"] == process_row["age"]]
    cands = cands[cands["diagnosis_norm"] == process_row["diagnosis_norm"]]

    # exact match on MMSE, but only enforced where available on both sides
    if pd.notna(process_row["MMSE"]):
        cands = cands[cands["MMSE"] == process_row["MMSE"]]

    return cands


match_rows = []
candidates_by_process_id = {}

for _, prow in meta_process.iterrows():
    cands = get_candidates(prow, meta_cogno)
    candidates_by_process_id[prow["IDs"]] = cands

    match_rows.append({
        "process_ID": prow["IDs"],
        "n_candidates": len(cands),
        "cogno_IDs": list(cands["dir_name"]),
    })

match_results = pd.DataFrame(match_rows)
match_results.head()

,process_ID,n_candidates,cogno_IDs
0,PROCESS-2_rec__001,21,"[MND_S_79150_251026_100527, R_25431_230228_111..."
1,PROCESS-2_rec__002,24,"[MND_R_61609_251111_125741, MND_S_01007_251026..."
2,PROCESS-2_rec__003,2,"[R_06515_240921_125238, R_21794_231109_102031]"
3,PROCESS-2_rec__004,31,"[MND_S_28481_251026_134914, MND_S_61039_251025..."
4,PROCESS-2_rec__005,21,"[MND_S_79150_251026_100527, R_25431_230228_111..."


In [151]:
# how confident are the matches?
# - unique: exactly one exact-match candidate -> take it as matched
# - ambiguous: 2+ exact-match candidates -> needs WER-based disambiguation
# - no_candidates: nothing survived the exact-match filters -> needs WER-based
#   disambiguation against a looser candidate pool, or manual review

def classify(n):
    if n == 0:
        return "no_candidates"
    return "unique" if n == 1 else "ambiguous"

match_results["status"] = match_results["n_candidates"].apply(classify)
print(match_results["status"].value_counts())

status
ambiguous        270
unique            92
no_candidates     38
Name: count, dtype: int64


In [152]:
# print list with n_candidates == 1
match_results[match_results["n_candidates"] == 1]

,process_ID,n_candidates,cogno_IDs,status
5,PROCESS-2_rec__006,1,[R_80598_241208_154228],unique
7,PROCESS-2_rec__008,1,[R_10609_231218_222057],unique
9,PROCESS-2_rec__010,1,[R_97637_250130_114826],unique
15,PROCESS-2_rec__016,1,[R_47809_241120_201151],unique
17,PROCESS-2_rec__018,1,[R_42610_241021_122231],unique
...,...,...,...,...
387,PROCESS-2_rec__388,1,[R_08987_241128_173712],unique
389,PROCESS-2_rec__390,1,[R_88348_230718_103115],unique
391,PROCESS-2_rec__392,1,[S_46707_240818_120203],unique
392,PROCESS-2_rec__393,1,[R_86157_241003_181142],unique


## Sanity check: do the PFT transcripts actually agree?

For every Process2 participant with a **unique** exact match, compare the PFT `.txt` transcript
on both sides as a sanity check that the metadata match is correct:

- Process2: `data/raw/text/PFT/{process_ID}__PFT.txt`
- CognoSpeak-new: `{base_dir}/{dir_name}/{dir_name}.Q10.txt`, where `base_dir` is one of
  **two** possible directories within CognoSpeak-new itself — fill both into
  `COGNO_NEW_BASE_DIRS` below; each `dir_name` is looked up in whichever of the two actually
  contains it.</cell id="placeholder">

In [153]:
from pathlib import Path

PROCESS_PFT_DIR = "/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/raw/text/PFT"

COGNO_NEW_BASE_DIRS = ["/mnt/xdrive/cchat/Shared/COGNO_DATA_SHARE/Cogno_ASR_trans",
                        "/mnt/xdrive/cchat/Shared/COGNO_DATA_SHARE/Cogno_MAN_trans"]


def process_pft_path(process_id):
    return Path(PROCESS_PFT_DIR) / f"{process_id}__PFT.txt"


def cogno_pft_path(dir_name, base_dirs=COGNO_NEW_BASE_DIRS):
    """Look up dir_name's PFT transcript across the candidate base dirs. Some
    transcripts have been manually edited, saved alongside the original with a
    "_A" suffix (*__PFT_A.txt) -- prefer that edited version over the plain
    *__PFT.txt when it exists. Glob rather than a fixed f-string because the
    on-disk filename stem doesn't always match dir_name exactly (an "MND_"
    prefix is dropped inconsistently between the two base dirs)."""
    for base_dir in base_dirs:
        folder = Path(base_dir) / dir_name
        if not folder.is_dir():
            continue
        adjusted = sorted(folder.glob("*__PFT_A.txt"))
        if adjusted:
            return adjusted[0]
        plain = sorted(folder.glob("*__PFT.txt"))
        if plain:
            return plain[0]
    # not found anywhere; return a path under the first base dir so callers
    # still have something to report as missing
    return Path(base_dirs[0]) / dir_name / f"{dir_name}__PFT.txt"


def read_text(path):
    if not path.exists():
        return None
    return path.read_text(encoding="utf-8", errors="replace")

In [154]:
unique_matches = match_results[match_results["status"] == "unique"].copy()
unique_matches["cogno_dir_name"] = unique_matches["cogno_IDs"].apply(lambda ids: ids[0])

sanity_rows = []
for _, row in unique_matches.iterrows():
    p_text = read_text(process_pft_path(row["process_ID"]))
    c_text = read_text(cogno_pft_path(row["cogno_dir_name"], base_dirs=COGNO_NEW_BASE_DIRS))

    sanity_rows.append({
        "process_ID": row["process_ID"],
        "cogno_dir_name": row["cogno_dir_name"],
        "process_file_exists": p_text is not None,
        "cogno_file_exists": c_text is not None,
        "text_match": (
            p_text is not None
            and c_text is not None
            and p_text.strip() == c_text.strip()
        ),
    })

sanity_results = pd.DataFrame(sanity_rows)
sanity_results.head(20)

,process_ID,cogno_dir_name,process_file_exists,cogno_file_exists,text_match
0,PROCESS-2_rec__006,R_80598_241208_154228,True,True,True
1,PROCESS-2_rec__008,R_10609_231218_222057,True,False,False
2,PROCESS-2_rec__010,R_97637_250130_114826,True,True,True
3,PROCESS-2_rec__016,R_47809_241120_201151,True,True,True
4,PROCESS-2_rec__018,R_42610_241021_122231,True,True,False
5,PROCESS-2_rec__029,R_72524_231122_170343,True,False,False
6,PROCESS-2_rec__039,R_07193_250423_155146,True,True,False
7,PROCESS-2_rec__044,R_13650_250407_120948,True,True,True
8,PROCESS-2_rec__048,R_03479_231113_144316,True,False,False
9,PROCESS-2_rec__054,R_85639_240926_111125,True,True,True


In [155]:
print("both files found:", (sanity_results["process_file_exists"] & sanity_results["cogno_file_exists"]).sum(), "/", len(sanity_results))
print()
print(sanity_results.groupby(["process_file_exists", "cogno_file_exists", "text_match"]).size())
print()
print("mismatches to inspect:")
sanity_results[sanity_results["process_file_exists"] & sanity_results["cogno_file_exists"] & ~sanity_results["text_match"]]

both files found: 54 / 92

process_file_exists  cogno_file_exists  text_match
True                 False              False         38
                     True               False         14
                                        True          40
dtype: int64

mismatches to inspect:


,process_ID,cogno_dir_name,process_file_exists,cogno_file_exists,text_match
4,PROCESS-2_rec__018,R_42610_241021_122231,True,True,False
6,PROCESS-2_rec__039,R_07193_250423_155146,True,True,False
15,PROCESS-2_rec__077,R_08145_241126_160159,True,True,False
20,PROCESS-2_rec__089,R_72606_250630_140717,True,True,False
40,PROCESS-2_rec__169,R_53515_240708_104553,True,True,False
42,PROCESS-2_rec__174,R_52347_240903_220336,True,True,False
47,PROCESS-2_rec__198,R_36086_240821_100924,True,True,False
55,PROCESS-2_rec__227,R_01077_240504_200709,True,True,False
61,PROCESS-2_rec__237,R_52347_240903_220336,True,True,False
63,PROCESS-2_rec__242,R_88710_241022_134020,True,True,False


## Alternative approach: match directly on PFT transcript content (CognoSpeak-new only)

Rather than filtering candidates by demographics first, this instead compares every Process2 PFT
transcript against **every** CognoSpeak-new PFT transcript that exists on disk (under either
`COGNO_NEW_BASE_DIRS`, preferring the edited `*__PFT_A.txt` version — same lookup as above) and
takes the lowest-WER (word error rate) one as the match. This sidesteps demographic-field
reliability entirely and naturally limits the search to participants who actually have a PFT
transcript file.

Timestamps like `(5 seconds)` and the leading speaker label (e.g. `Pat:`) are stripped before
comparing, since those are annotation artifacts rather than content.</cell id="placeholder">

In [156]:
import re
import jiwer

def normalize_transcript(text):
    text = re.sub(r"\([^)]*\)", " ", text)       # drop "(5 seconds)" style annotations
    text = re.sub(r"\b\w+:\s*", " ", text)        # drop "Pat:" / "Ass:" style speaker labels
    text = re.sub(r"[^\w\s]", " ", text)          # drop punctuation
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


# load every Process2 PFT transcript
process_texts = {}
for process_id in meta_process["IDs"]:
    text = read_text(process_pft_path(process_id))
    if text is not None:
        process_texts[process_id] = normalize_transcript(text)

# load every CognoSpeak-new PFT transcript that actually exists on disk under
# either base dir (independent of the metadata match above)
cogno_dir_names = set()
for base_dir in COGNO_NEW_BASE_DIRS:
    cogno_dir_names.update(p.name for p in Path(base_dir).iterdir() if p.is_dir())

cogno_texts = {}
for dir_name in cogno_dir_names:
    text = read_text(cogno_pft_path(dir_name))
    if text is not None:
        cogno_texts[dir_name] = normalize_transcript(text)

print(f"{len(process_texts)}/{len(meta_process)} Process2 PFT transcripts found")
print(f"{len(cogno_texts)}/{len(cogno_dir_names)} CognoSpeak-new PFT transcripts found")

400/400 Process2 PFT transcripts found
286/286 CognoSpeak-new PFT transcripts found


In [157]:
# for every Process2 transcript, find the lowest-WER CognoSpeak-new transcript
transcript_match_rows = []

for process_id, ptext in process_texts.items():
    wers = {
        cogno_dir: jiwer.wer(ptext or " ", ctext or " ")
        for cogno_dir, ctext in cogno_texts.items()
    }
    ranked = sorted(wers.items(), key=lambda kv: kv[1])

    best_dir, best_wer = ranked[0]
    second_wer = ranked[1][1] if len(ranked) > 1 else np.nan

    transcript_match_rows.append({
        "process_ID": process_id,
        "best_cogno_dir": best_dir,
        "best_wer": best_wer,
        "second_best_wer": second_wer,
        "margin": second_wer - best_wer,
    })

transcript_match_results = pd.DataFrame(transcript_match_rows)
transcript_match_results.sort_values("best_wer").head(10)

,process_ID,best_cogno_dir,best_wer,second_best_wer,margin
399,PROCESS-2_rec__400,R_32882_240408_202754,0.0,1.000000,1.000000
396,PROCESS-2_rec__397,S_23585_250419_151846,0.0,1.000000,1.000000
394,PROCESS-2_rec__395,S_16195_240913_195012,0.0,0.894737,0.894737
32,PROCESS-2_rec__033,S_72789_240119_150806,0.0,0.941176,0.941176
33,PROCESS-2_rec__034,S_28185_240128_134059,0.0,0.862069,0.862069
34,PROCESS-2_rec__035,S_65348_240111_101715,0.0,0.916667,0.916667
36,PROCESS-2_rec__037,S_60372_240404_121617,0.0,0.923077,0.923077
37,PROCESS-2_rec__038,S_83009_231027_165439,0.0,0.882353,0.882353
38,PROCESS-2_rec__039,R_07193_250423_155146,0.0,0.888889,0.888889
43,PROCESS-2_rec__044,R_13650_250407_120948,0.0,0.888889,0.888889


In [158]:
# confidence: a low best_wer with a decent margin to the second-best candidate
# is a strong, unambiguous transcript match
CONFIDENT_WER = 0.15   # near-exact transcript
CONFIDENT_MARGIN = 0.15  # clearly better than the runner-up

transcript_match_results["confident"] = (
    (transcript_match_results["best_wer"] <= CONFIDENT_WER)
    & (transcript_match_results["margin"] >= CONFIDENT_MARGIN)
)

print(transcript_match_results["confident"].value_counts())
print()
print("best_wer distribution:")
print(transcript_match_results["best_wer"].describe())

confident
False    203
True     197
Name: count, dtype: int64

best_wer distribution:
count    400.000000
mean       0.397881
std        0.418196
min        0.000000
25%        0.000000
50%        0.166957
75%        0.873680
max        1.000000
Name: best_wer, dtype: float64


In [159]:
# cross-check: how often does the confident transcript match agree with the
# earlier demographic-based "unique" match, where both exist?
comparison = transcript_match_results.merge(
    match_results[["process_ID", "status", "cogno_IDs"]], on="process_ID", how="left"
)
comparison["demo_unique_dir"] = comparison["cogno_IDs"].apply(
    lambda ids: ids[0] if isinstance(ids, list) and len(ids) == 1 else None
)

both = comparison[comparison["confident"] & comparison["demo_unique_dir"].notna()]
agree = both["best_cogno_dir"] == both["demo_unique_dir"]

print(f"{agree.sum()}/{len(both)} agree, where both a confident transcript match and a unique demographic match exist")
print()
print("disagreements (worth a manual look):")
both[~agree][["process_ID", "best_cogno_dir", "demo_unique_dir", "best_wer", "margin"]]

47/49 agree, where both a confident transcript match and a unique demographic match exist

disagreements (worth a manual look):


,process_ID,best_cogno_dir,demo_unique_dir,best_wer,margin
282,PROCESS-2_rec__283,MND_R_15193_240807_142546,R_15193_240807_142546,0.0,0.928571
283,PROCESS-2_rec__284,R_83691_250414_104757,R_06126_250723_181823,0.0,0.909091


## Combine PFT + SFT + CTD for stronger transcript matching

PFT alone left 203/400 participants unconfident. Process2 also has SFT (semantic fluency) and CTD
(Cookie Theft description) transcripts, and the same CognoSpeak-new folders contain `__SFT`/`__CTD`
files alongside `__PFT`. Bringing all three into the WER comparison gives three independent looks
at the same person, which should sharpen genuine matches and make coincidental low-WER ties on a
single short task much less likely.

Note CTD is spontaneous picture description rather than a word list, so its WER for a genuine
match will run noticeably higher than PFT/SFT's — the per-task WERs are kept alongside the
combined score below so the confidence threshold can be sanity-checked rather than assumed.</cell id="placeholder">

In [160]:
PROCESS_TEXT_DIR = "/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/raw/text"
TASKS = ["PFT", "SFT", "CTD"]


def process_task_path(process_id, task):
    return Path(PROCESS_TEXT_DIR) / task / f"{process_id}__{task}.txt"


def cogno_task_path(dir_name, task, base_dirs=COGNO_NEW_BASE_DIRS):
    """Same lookup as cogno_pft_path, generalized to any task suffix."""
    for base_dir in base_dirs:
        folder = Path(base_dir) / dir_name
        if not folder.is_dir():
            continue
        adjusted = sorted(folder.glob(f"*__{task}_A.txt"))
        if adjusted:
            return adjusted[0]
        plain = sorted(folder.glob(f"*__{task}.txt"))
        if plain:
            return plain[0]
    return Path(base_dirs[0]) / dir_name / f"{dir_name}__{task}.txt"


# load every Process2 and CognoSpeak-new transcript, per task
process_texts_by_task = {task: {} for task in TASKS}
cogno_texts_by_task = {task: {} for task in TASKS}

for task in TASKS:
    for process_id in meta_process["IDs"]:
        text = read_text(process_task_path(process_id, task))
        if text is not None:
            process_texts_by_task[task][process_id] = normalize_transcript(text)

    for dir_name in cogno_dir_names:
        text = read_text(cogno_task_path(dir_name, task))
        if text is not None:
            cogno_texts_by_task[task][dir_name] = normalize_transcript(text)

    print(f"{task}: {len(process_texts_by_task[task])}/{len(meta_process)} process, "
          f"{len(cogno_texts_by_task[task])}/{len(cogno_dir_names)} cogno")

PFT: 400/400 process, 286/286 cogno
SFT: 400/400 process, 286/286 cogno
CTD: 400/400 process, 286/286 cogno


In [161]:
def compute_task_wers(process_texts, cogno_texts):
    """{process_id: {cogno_dir: wer}} for every process_id x cogno_dir pair
    where both texts exist for this task."""
    return {
        process_id: {
            cogno_dir: jiwer.wer(ptext or " ", ctext or " ")
            for cogno_dir, ctext in cogno_texts.items()
        }
        for process_id, ptext in process_texts.items()
    }


wer_by_task = {
    task: compute_task_wers(process_texts_by_task[task], cogno_texts_by_task[task])
    for task in TASKS
}


combined_match_rows = []

for process_id in meta_process["IDs"]:
    candidate_dirs = set()
    for task in TASKS:
        candidate_dirs.update(wer_by_task[task].get(process_id, {}).keys())
    if not candidate_dirs:
        continue

    scored = []
    for cogno_dir in candidate_dirs:
        per_task = {
            task: wer_by_task[task][process_id][cogno_dir]
            for task in TASKS
            if process_id in wer_by_task[task] and cogno_dir in wer_by_task[task][process_id]
        }
        combined_wer = sum(per_task.values()) / len(per_task)
        scored.append((cogno_dir, combined_wer, per_task))

    scored.sort(key=lambda x: x[1])
    best_dir, best_wer, best_per_task = scored[0]
    second_wer = scored[1][1] if len(scored) > 1 else np.nan

    combined_match_rows.append({
        "process_ID": process_id,
        "best_cogno_dir": best_dir,
        "best_combined_wer": best_wer,
        "n_tasks_compared": len(best_per_task),
        "pft_wer": best_per_task.get("PFT"),
        "sft_wer": best_per_task.get("SFT"),
        "ctd_wer": best_per_task.get("CTD"),
        "second_best_wer": second_wer,
        "margin": second_wer - best_wer,
    })

combined_match_results = pd.DataFrame(combined_match_rows)
combined_match_results.sort_values("best_combined_wer").head(10)

,process_ID,best_cogno_dir,best_combined_wer,n_tasks_compared,pft_wer,sft_wer,ctd_wer,second_best_wer,margin
21,PROCESS-2_rec__022,S_17759_250103_193434,0.0,3,0.0,0.0,0.0,0.828327,0.828327
20,PROCESS-2_rec__021,S_68401_240721_175629,0.0,3,0.0,0.0,0.0,0.898148,0.898148
43,PROCESS-2_rec__044,R_13650_250407_120948,0.0,3,0.0,0.0,0.0,0.921765,0.921765
23,PROCESS-2_rec__024,R_76728_250625_115446,0.0,3,0.0,0.0,0.0,0.913512,0.913512
357,PROCESS-2_rec__358,R_01541_250208_144443,0.0,3,0.0,0.0,0.0,0.880059,0.880059
354,PROCESS-2_rec__355,R_70344_240306_115555,0.0,3,0.0,0.0,0.0,0.892274,0.892274
45,PROCESS-2_rec__046,S_23922_240604_201637,0.0,3,0.0,0.0,0.0,0.856380,0.856380
44,PROCESS-2_rec__045,R_15869_250204_113026,0.0,3,0.0,0.0,0.0,0.900317,0.900317
360,PROCESS-2_rec__361,S_54295_231107_175543,0.0,3,0.0,0.0,0.0,0.917053,0.917053
255,PROCESS-2_rec__256,R_29207_250626_103436,0.0,3,0.0,0.0,0.0,0.888889,0.888889


In [162]:
# reuse the same confidence thresholds as the PFT-only pass as a starting point --
# check the per-task WER columns above before trusting this blindly, since CTD's
# higher baseline WER will pull the combined score up even for true matches
combined_match_results["confident"] = (
    (combined_match_results["best_combined_wer"] <= CONFIDENT_WER)
    & (combined_match_results["margin"] >= CONFIDENT_MARGIN)
)

print(combined_match_results["confident"].value_counts())
print()
print("n_tasks_compared distribution:")
print(combined_match_results["n_tasks_compared"].value_counts())
print()
print("PFT-only confident vs. combined confident, for the same process_IDs:")
cmp = transcript_match_results[["process_ID", "confident"]].rename(columns={"confident": "pft_only_confident"}).merge(
    combined_match_results[["process_ID", "confident"]].rename(columns={"confident": "combined_confident"}),
    on="process_ID",
)
print(pd.crosstab(cmp["pft_only_confident"], cmp["combined_confident"]))

confident
True     201
False    199
Name: count, dtype: int64

n_tasks_compared distribution:
n_tasks_compared
3    400
Name: count, dtype: int64

PFT-only confident vs. combined confident, for the same process_IDs:
combined_confident  False  True 
pft_only_confident              
False                 193     10
True                    6    191


## Split into a sanity-check subset and a to-investigate subset

- **`any_zero_wer_matches`**: the best candidate has at least one task (PFT/SFT/CTD) with WER
  exactly 0 — i.e. a word-for-word transcript match on at least one task. These are the ones to
  eyeball as a sanity check, since an exact match on even one task is very unlikely by chance.
- **`no_zero_wer_matches`**: none of the three tasks hit WER 0 for the best candidate. These are
  the remaining/uncertain participants — worth checking whether they genuinely have no match in
  CognoSpeak-new (no transcript file, or truly absent) or whether the best candidate is still
  right despite the edits/trimming mentioned in the intro.</cell id="placeholder">

In [163]:
task_wer_cols = ["pft_wer", "sft_wer", "ctd_wer"]

combined_match_results["any_task_wer_zero"] = (
    combined_match_results[task_wer_cols].eq(0).any(axis=1)
)

any_zero_wer_matches = combined_match_results[combined_match_results["any_task_wer_zero"]].copy()
no_zero_wer_matches = combined_match_results[~combined_match_results["any_task_wer_zero"]].copy()

print(f"any_zero_wer_matches: {len(any_zero_wer_matches)}")
print(f"no_zero_wer_matches: {len(no_zero_wer_matches)}")

any_zero_wer_matches: 190
no_zero_wer_matches: 210


In [164]:
any_zero_wer_matches.sort_values("best_combined_wer")

,process_ID,best_cogno_dir,best_combined_wer,n_tasks_compared,pft_wer,sft_wer,ctd_wer,second_best_wer,margin,confident,any_task_wer_zero
0,PROCESS-2_rec__001,S_51399_231218_222617,0.000000,3,0.000000,0.000000,0.000000,0.908221,0.908221,True,True
1,PROCESS-2_rec__002,S_69660_240213_152129,0.000000,3,0.000000,0.000000,0.000000,0.921310,0.921310,True,True
2,PROCESS-2_rec__003,R_06515_240921_125238,0.000000,3,0.000000,0.000000,0.000000,0.889971,0.889971,True,True
5,PROCESS-2_rec__006,R_80598_241208_154228,0.000000,3,0.000000,0.000000,0.000000,0.906349,0.906349,True,True
12,PROCESS-2_rec__013,S_62026_240612_094236,0.000000,3,0.000000,0.000000,0.000000,0.867958,0.867958,True,True
...,...,...,...,...,...,...,...,...,...,...,...
358,PROCESS-2_rec__359,S_75369_250710_111319,0.136659,3,0.000000,0.333333,0.076642,0.891708,0.755050,True,True
3,PROCESS-2_rec__004,S_20220_231116_194558,0.144354,3,0.000000,0.344828,0.088235,1.045639,0.901285,True,True
395,PROCESS-2_rec__396,S_79141_250424_135702,0.159420,3,0.173913,0.304348,0.000000,0.899476,0.740056,False,True
10,PROCESS-2_rec__011,R_23893_230630_100313,0.271605,3,0.000000,0.814815,0.000000,0.945006,0.673401,False,True


In [165]:
no_zero_wer_matches.sort_values("best_combined_wer")

,process_ID,best_cogno_dir,best_combined_wer,n_tasks_compared,pft_wer,sft_wer,ctd_wer,second_best_wer,margin,confident,any_task_wer_zero
361,PROCESS-2_rec__362,S_71946_250120_143248,0.043474,3,0.028571,0.037037,0.064815,0.888889,0.845414,True,False
116,PROCESS-2_rec__117,S_46931_231117_125407,0.046296,3,0.055556,0.062500,0.020833,0.890625,0.844329,True,False
348,PROCESS-2_rec__349,S_77509_231214_160531,0.073908,3,0.131579,0.040816,0.049327,0.901084,0.827177,True,False
327,PROCESS-2_rec__328,S_12853_231121_111156,0.075250,3,0.100000,0.093750,0.032000,0.885917,0.810667,True,False
55,PROCESS-2_rec__056,S_30180_231226_165304,0.077383,3,0.151515,0.057971,0.022663,0.899010,0.821627,True,False
...,...,...,...,...,...,...,...,...,...,...,...
24,PROCESS-2_rec__025,S_20220_231116_194558,1.065464,3,0.937500,0.954545,1.304348,1.230567,0.165102,False,False
212,PROCESS-2_rec__213,R_32882_240408_202754,1.141176,3,1.000000,0.823529,1.600000,1.334118,0.192941,False,False
131,PROCESS-2_rec__132,R_32882_240408_202754,1.300945,3,1.000000,0.692308,2.210526,1.316175,0.015230,False,False
379,PROCESS-2_rec__380,S_20220_231116_194558,1.576923,3,1.230769,1.000000,2.500000,1.784870,0.207947,False,False


In [166]:
combined_match_results.to_csv("/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/processed/combined_match_results.csv", index=False)

## Extracting task answers from old CognoSpeak transcripts

Old CognoSpeak (the IVA and pre-2020 CognoSpeak systems) stores each participant as a single
`.txt` dialogue transcript containing answers to *all* questions asked in that session, rather
than one file per task. The system evolved over time (`iva1`, `iva2`, `iva3`, `ivamem_*`,
`ivastroke_*`, `cchat`, ...), so the exact question wording and the set of questions asked differ
between transcripts -- some versions don't include the picture description task at all, for
example.

`data/raw/PromptsMapping.csv` lists every question phrasing used by every system version. This
section only cares about the three tasks needed for Process2 matching:
- `semantic-fluency`
- `phonemic-fluency`
- `picture-description-cookietheft`

**Approach**: parse each transcript into speaker turns (lines are continued until the next
`Speaker:`-prefixed line), fuzzy-match every interviewer turn (`Int*`/`Ass*`/`Oth*`) against the
known phrasings for the three target tasks, and take the participant's response following the
*last* matching prompt (fluency prompts are sometimes repeated verbatim before the participant
actually answers). Transcripts missing any of the three tasks are dropped.

In [167]:
import re

PROMPTS_MAPPING_PATH = BASE_DIR + "PromptsMapping.csv"
TARGET_TASKS = ["semantic-fluency", "phonemic-fluency", "picture-description-cookietheft"]

prompts_mapping = pd.read_csv(PROMPTS_MAPPING_PATH)


def normalize_prompt(text):
    text = re.sub(r"[^\w\s]", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


# every known phrasing of each target task, across all IVA/CognoSpeak versions in the mapping
target_prompt_variants = {
    task: [
        normalize_prompt(t)
        for t in prompts_mapping.loc[prompts_mapping["PromptLongCode"] == task, "PromptText"].unique()
    ]
    for task in TARGET_TASKS
}

for task, variants in target_prompt_variants.items():
    print(f"{task}: {len(variants)} known phrasing(s)")

semantic-fluency: 5 known phrasing(s)
phonemic-fluency: 4 known phrasing(s)
picture-description-cookietheft: 4 known phrasing(s)


In [168]:
# --- speaker-turn parsing ---
# Old CognoSpeak transcripts use "Speaker: text" lines, where a turn's text can
# wrap onto following un-prefixed lines (or be separated by a blank line) until
# the next "Speaker:" line starts. Speakers seen in practice: Int1/Int2 (asking
# the questions), Oth (a third party), Pat (the participant answering).
SPEAKER_LINE_RE = re.compile(r"^([A-Za-z]+\d*)\s*:\s*(.*)$")
INTERVIEWER_LABEL_RE = re.compile(r"^(int|ass|oth)", re.IGNORECASE)


def parse_turns(text):
    """Return an ordered list of (speaker_label, turn_text) tuples."""
    turns = []
    label, lines = None, []

    def flush():
        if label is not None:
            turns.append((label, re.sub(r"\s+", " ", " ".join(lines)).strip()))

    for raw_line in text.splitlines():
        line = raw_line.strip()
        m = SPEAKER_LINE_RE.match(line)
        if m:
            flush()
            label, lines = m.group(1), [m.group(2)]
        elif line:
            lines.append(line)
    flush()
    return turns


# --- fuzzy prompt matching ---
# Word-overlap (Jaccard) on content words separates the three target prompts
# from every other question cleanly (~0.85-1.0 for true matches vs. <=0.2 for
# everything else), which is much more robust to the wording drift between
# system versions than a raw character-level ratio.
STOPWORDS = set(
    "the a an to and or of in on for as you can that this it is are with your "
    "please have has been many much give as detail possible".split()
)


def content_words(normalized_text):
    return {w for w in normalized_text.split() if w not in STOPWORDS and len(w) > 2}


def jaccard(a_words, b_words):
    if not a_words or not b_words:
        return 0.0
    return len(a_words & b_words) / len(a_words | b_words)


def best_match_score(turn_text, prompt_variants):
    words = content_words(normalize_prompt(turn_text))
    if not words:
        return 0.0
    return max(jaccard(words, content_words(v)) for v in prompt_variants)

In [ ]:
MATCH_THRESHOLD = 0.5  # true matches score ~0.85-1.0, next-best distractor tops out ~0.2

# Which occurrence to anchor on, per task, when an interviewer turn matches more
# than once:
# - fluency tasks: "last" -- the instructions are sometimes re-read verbatim
#   right before the participant actually starts (participant acknowledges the
#   first reading, e.g. "Mm, I've got that.", then it's repeated).
# - picture-description-cookietheft: "first" -- PromptsMapping.csv gives this
#   task and picture-description-complex1 (a *different* picture, asked
#   immediately afterwards) the exact same instruction text in every system
#   version, so they can't be told apart by wording. Cookie Theft is always
#   asked first, so the first occurrence is the correct one; taking the last
#   occurrence would silently grab the complex1 answer instead.
MATCH_ANCHOR = {
    "semantic-fluency": "last",
    "phonemic-fluency": "last",
    "picture-description-cookietheft": "first",
}


def extract_task_answers(turns, prompt_variants_by_task=target_prompt_variants,
                          threshold=MATCH_THRESHOLD, match_anchor=MATCH_ANCHOR):
    """For each target task, find the matching interviewer turn (per
    `match_anchor`) and return the participant's response: all non-interviewer
    turns following it, up to the next interviewer turn."""
    answers = {}
    for task, variants in prompt_variants_by_task.items():
        match_indices = [
            i
            for i, (label, text) in enumerate(turns)
            if INTERVIEWER_LABEL_RE.match(label) and best_match_score(text, variants) >= threshold
        ]
        if not match_indices:
            continue

        anchor_idx = match_indices[-1] if match_anchor.get(task, "last") == "last" else match_indices[0]
        answer_parts = []
        for label, text in turns[anchor_idx + 1:]:
            if INTERVIEWER_LABEL_RE.match(label):
                break
            if text:
                answer_parts.append(text)
        answers[task] = " ".join(answer_parts).strip()
    return answers


def process_old_cogno_transcripts(txt_dirs, tasks=TARGET_TASKS, threshold=MATCH_THRESHOLD):
    """Parse every .txt transcript under txt_dirs (a single directory, or a list
    of directories -- old CognoSpeak data is often split across several, e.g.
    an IVA folder and a separate CognoSpeak folder) and extract answers to `tasks`."""
    if isinstance(txt_dirs, (str, Path)):
        txt_dirs = [txt_dirs]

    rows = []
    for txt_dir in txt_dirs:
        for fpath in sorted(Path(txt_dir).glob("*.txt")):
            raw_text = read_text(fpath)
            if raw_text is None:
                continue
            turns = parse_turns(raw_text)
            answers = extract_task_answers(turns, threshold=threshold)

            row = {"file": fpath.name, "record_id": fpath.stem}
            for task in tasks:
                row[task] = answers.get(task)
            row["has_all_tasks"] = all(bool(row.get(task)) for task in tasks)
            rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# TODO: fill in the directory (or list of directories) containing the old
# CognoSpeak .txt transcripts, e.g.:
# OLD_COGNO_TXT_DIR = ["/path/to/IVA/TXT-files", "/path/to/CognoSpeak/TXT-files"]
OLD_COGNO_TXT_DIR = ""

old_cogno_answers = process_old_cogno_transcripts(OLD_COGNO_TXT_DIR)

print(old_cogno_answers["has_all_tasks"].value_counts())
old_cogno_answers.head()

there seems to be a small problem with the code. for example 268-H004-2020-12-12-11-58-17 only returns 'Ok' for PFT because the prompt is split by that 'Ok' from the patient. the answer comes after the second hald of prompt is finished.

**TO DO**: inspect all cases where has_all_tasks == False 

In [172]:
# keep only transcripts where all three target tasks were found
old_cogno_complete = old_cogno_answers[old_cogno_answers["has_all_tasks"]].drop(columns=["has_all_tasks"])
print(f"{len(old_cogno_complete)}/{len(old_cogno_answers)} transcripts have all three tasks")

#old_cogno_complete.to_csv(
#    "/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/processed/old_cognospeak_task_answers.csv",
#    index=False,
#)
old_cogno_complete.head()

189/207 transcripts have all three tasks


,file,record_id,semantic-fluency,phonemic-fluency,picture-description-cookietheft
0,219-M001-2020-11-13-11-12-10.txt,219-M001-2020-11-13-11-12-10,"Giraffe, lion, gibbon, monkey, baboon (2 secon...","People, precious, pipe (1 second) particular (...","(3 seconds) Er, it looks like a sunny day, the..."
2,222-M004-2020-11-20-14-10-52.txt,222-M004-2020-11-20-14-10-52,"Um (3 seconds) (buzzer sounds) er cat, dog, go...","Pepper (buzzer sounds) oh, pepper, parsley, oh...","Er, there’s, there’s a woman standing at the k..."
3,223-M005-2020-11-17-11-9-42.txt,223-M005-2020-11-17-11-9-42,"Dog, cat, rabbit, cow, horse, lion, tiger, gir...","Pipe, porter, pewter, er, p, p, er (1 second) ...",Well it’s a kitchen where the mother is (cough...
4,224-M006-2020-12-3-13-29-1.txt,224-M006-2020-12-3-13-29-1,"Cow, sheep, pig, goat, rabbit, hare, pheasant,...","(Coughs) Pot, pan, pass, passive, past, person...","This is a picture of a kitchen, domestic scene..."
5,225-M007-2020-11-17-15-43-26.txt,225-M007-2020-11-17-15-43-26,"(Sighs) Cat, dog, gerbil, lion, tiger, um elep...","Pepper, persistence, um pints, er palindrome, ...","OK. Um, on the left-hand side of the picture a..."


## Match old CognoSpeak transcripts to Process2, the same way as CognoSpeak-new

Reuses the combined PFT+SFT+CTD WER-matching approach from above (`normalize_transcript`,
`compute_task_wers`, same combined-score/confidence logic), just swapping the CognoSpeak-new
per-task texts for the ones just extracted from the old CognoSpeak transcripts in
`old_cogno_complete`. `process_texts_by_task` (Process2's PFT/SFT/CTD texts) is already loaded
from that earlier section, so it doesn't need to be rebuilt.

Task name mapping: `semantic-fluency` -> SFT, `phonemic-fluency` -> PFT,
`picture-description-cookietheft` -> CTD.

In [ ]:
OLD_COGNO_TASK_MAP = {
    "semantic-fluency": "SFT",
    "phonemic-fluency": "PFT",
    "picture-description-cookietheft": "CTD",
}

# {process_task: {record_id: normalized_text}}, mirroring cogno_texts_by_task above
old_cogno_texts_by_task = {task: {} for task in TASKS}

for old_col, task in OLD_COGNO_TASK_MAP.items():
    for _, row in old_cogno_complete.iterrows():
        text = row[old_col]
        if text:
            old_cogno_texts_by_task[task][row["record_id"]] = normalize_transcript(text)

for task in TASKS:
    print(f"{task}: {len(old_cogno_texts_by_task[task])} old-CognoSpeak transcripts")

In [ ]:
old_cogno_wer_by_task = {
    task: compute_task_wers(process_texts_by_task[task], old_cogno_texts_by_task[task])
    for task in TASKS
}

old_cogno_match_rows = []

for process_id in meta_process["IDs"]:
    candidate_records = set()
    for task in TASKS:
        candidate_records.update(old_cogno_wer_by_task[task].get(process_id, {}).keys())
    if not candidate_records:
        continue

    scored = []
    for record_id in candidate_records:
        per_task = {
            task: old_cogno_wer_by_task[task][process_id][record_id]
            for task in TASKS
            if process_id in old_cogno_wer_by_task[task] and record_id in old_cogno_wer_by_task[task][process_id]
        }
        combined_wer = sum(per_task.values()) / len(per_task)
        scored.append((record_id, combined_wer, per_task))

    scored.sort(key=lambda x: x[1])
    best_record, best_wer, best_per_task = scored[0]
    second_wer = scored[1][1] if len(scored) > 1 else np.nan

    old_cogno_match_rows.append({
        "process_ID": process_id,
        "best_old_cogno_record": best_record,
        "best_combined_wer": best_wer,
        "n_tasks_compared": len(best_per_task),
        "pft_wer": best_per_task.get("PFT"),
        "sft_wer": best_per_task.get("SFT"),
        "ctd_wer": best_per_task.get("CTD"),
        "second_best_wer": second_wer,
        "margin": second_wer - best_wer,
    })

old_cogno_match_results = pd.DataFrame(old_cogno_match_rows)

# same confidence + zero-WER sanity criteria as the CognoSpeak-new matching above
old_cogno_match_results["confident"] = (
    (old_cogno_match_results["best_combined_wer"] <= CONFIDENT_WER)
    & (old_cogno_match_results["margin"] >= CONFIDENT_MARGIN)
)
old_cogno_match_results["any_task_wer_zero"] = old_cogno_match_results[task_wer_cols].eq(0).any(axis=1)

print(old_cogno_match_results["confident"].value_counts())
print()
print(old_cogno_match_results["any_task_wer_zero"].value_counts())
old_cogno_match_results.sort_values("best_combined_wer").head(10)

### Combine new- and old-CognoSpeak matches into one best-source table

Every Process2 participant now has up to two independent candidate matches: one against
CognoSpeak-new (`combined_match_results`) and one against old CognoSpeak (`old_cogno_match_results`).
Take whichever source gives the lower combined WER as the overall match, and record which source
that was.

In [ ]:
new_side = combined_match_results[
    ["process_ID", "best_cogno_dir", "best_combined_wer", "margin", "confident", "any_task_wer_zero"]
].rename(columns={"best_cogno_dir": "best_match_id"})
new_side["source"] = "cognospeak_new"

old_side = old_cogno_match_results[
    ["process_ID", "best_old_cogno_record", "best_combined_wer", "margin", "confident", "any_task_wer_zero"]
].rename(columns={"best_old_cogno_record": "best_match_id"})
old_side["source"] = "cognospeak_old"

overall_matches = (
    pd.concat([new_side, old_side], ignore_index=True)
    .sort_values("best_combined_wer")
    .drop_duplicates("process_ID", keep="first")
    .sort_values("process_ID")
    .reset_index(drop=True)
)

print(overall_matches["source"].value_counts())
print()
print(overall_matches["confident"].value_counts())

overall_matches.to_csv(
    "/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/processed/overall_match_results.csv",
    index=False,
)
overall_matches.head(10)

### Look up the full ranked candidate list (incl. second-best) for specific Process2 IDs

The matching loops above only kept each process_ID's best candidate (plus the *value* of the
second-best WER, for the confidence margin) and threw away which candidate that second-best one
actually was. `ranked_matches` rebuilds the full ranked candidate list for one process_ID on
demand -- across **both** CognoSpeak-new and old CognoSpeak -- reusing the per-task WER dictionaries
already computed above (`wer_by_task`, `old_cogno_wer_by_task`), so it's cheap and doesn't
recompute any WERs.

In [ ]:
def _scored_candidates(process_id, wer_by_task_dict, source_label):
    candidate_ids = set()
    for task in TASKS:
        candidate_ids.update(wer_by_task_dict[task].get(process_id, {}).keys())

    rows = []
    for cand_id in candidate_ids:
        per_task = {
            task: wer_by_task_dict[task][process_id][cand_id]
            for task in TASKS
            if process_id in wer_by_task_dict[task] and cand_id in wer_by_task_dict[task][process_id]
        }
        rows.append({
            "process_ID": process_id,
            "candidate_id": cand_id,
            "source": source_label,
            "combined_wer": sum(per_task.values()) / len(per_task),
            **per_task,
        })
    return rows


def ranked_matches(process_id, n=None):
    """Every candidate for process_id, from both CognoSpeak-new and old
    CognoSpeak, ranked by combined WER ascending. rank 0 = best match (what the
    matching loops above kept), rank 1 = second-best, etc. Pass n to keep only
    the top n rows."""
    rows = (
        _scored_candidates(process_id, wer_by_task, "cognospeak_new")
        + _scored_candidates(process_id, old_cogno_wer_by_task, "cognospeak_old")
    )
    df = pd.DataFrame(rows).sort_values("combined_wer").reset_index(drop=True)
    df.insert(0, "rank", df.index)
    return df.head(n) if n else df


# fill in the process_IDs you want to inspect
PROCESS_IDS_TO_CHECK = ["PROCESS-2_rec__018", "PROCESS-2_rec__039"]

ranked_lookup = pd.concat([ranked_matches(pid, n=5) for pid in PROCESS_IDS_TO_CHECK], ignore_index=True)
ranked_lookup

## Try a third transcript source for the participants still unmatched

`data/raw/Process_reverse_matching_checked.csv` has since been manually checked: its `correct`
column records, per Process2 participant, whether the automated best match (from CognoSpeak-new
and/or old CognoSpeak above) was actually correct. This section targets just the participants
where `correct == "no"` and tries a third transcript source against them.

That third source has a different layout to both sources used so far -- nested one folder per
`dir_name`, with per-task files named by CognoSpeak question code rather than task abbreviation:

```
{base_dir}/{dir_name}/{dir_name}__Q10.txt   <- SFT (semantic fluency)
{base_dir}/{dir_name}/{dir_name}__Q11.txt   <- PFT (phonemic fluency)
{base_dir}/{dir_name}/{dir_name}__Q12.txt   <- CTD (picture description / Cookie Theft)
```

Fill in `THIRD_SOURCE_BASE_DIR` below before running. Everything else (`normalize_transcript`,
`TASKS`, `process_texts_by_task`, `_scored_candidates`, `CONFIDENT_WER`/`CONFIDENT_MARGIN`) reuses
what's already defined earlier in this notebook.

In [189]:
process_reverse_matching = pd.read_csv(BASE_DIR + "Process_reverse_matching_checked.csv")

# participants whose automated best match was manually marked wrong -- these
# are the ones this section tries to (re-)match against the third source
UNMATCHED_PROCESS_IDS = process_reverse_matching.loc[
    process_reverse_matching["correct"].str.lower() == "no", "process_ID"
].tolist()

print(f"{len(UNMATCHED_PROCESS_IDS)}/{len(process_reverse_matching)} participants still need a match")

102/400 participants still need a match


In [191]:
# TODO: fill in the base directory for the third transcript source
THIRD_SOURCE_BASE_DIR = "/mnt/xdrive/cchat/Shared/CognoSpeak_NEW/MANUAL_TRANSCRIPTS_PER_QUESTION"

# task -> CognoSpeak question code used in this source's filenames
THIRD_SOURCE_TASK_TO_Q = {"SFT": "Q10", "PFT": "Q11", "CTD": "Q12"}


def third_source_task_path(dir_name, task, base_dir=THIRD_SOURCE_BASE_DIR):
    q_code = THIRD_SOURCE_TASK_TO_Q[task]
    return Path(base_dir) / dir_name / f"{dir_name}_{q_code}.txt"


def find_dir_names(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.is_dir():
        return set()
    return {p.name for p in base_dir.iterdir() if p.is_dir()}


third_source_dir_names = find_dir_names(THIRD_SOURCE_BASE_DIR)

# {task: {dir_name: normalized_text}}, mirroring cogno_texts_by_task /
# old_cogno_texts_by_task above
third_source_texts_by_task = {task: {} for task in TASKS}
for task in TASKS:
    for dir_name in third_source_dir_names:
        text = read_text(third_source_task_path(dir_name, task))
        if text is not None:
            third_source_texts_by_task[task][dir_name] = normalize_transcript(text)

print(f"{len(third_source_dir_names)} participant folders found under THIRD_SOURCE_BASE_DIR")
for task in TASKS:
    print(f"{task}: {len(third_source_texts_by_task[task])} third-source transcripts found")

601 participant folders found under THIRD_SOURCE_BASE_DIR
PFT: 601 third-source transcripts found
SFT: 601 third-source transcripts found
CTD: 601 third-source transcripts found


In [192]:
def compute_task_wers_subset(process_texts, other_texts, process_ids):
    """Same as compute_task_wers above, but only for the given process_ids --
    no need to re-score participants that already have a confirmed match."""
    return {
        process_id: {
            other_id: jiwer.wer(process_texts[process_id] or " ", other_text or " ")
            for other_id, other_text in other_texts.items()
        }
        for process_id in process_ids
        if process_id in process_texts
    }


third_source_wer_by_task = {
    task: compute_task_wers_subset(process_texts_by_task[task], third_source_texts_by_task[task], UNMATCHED_PROCESS_IDS)
    for task in TASKS
}

third_source_match_rows = []
for process_id in UNMATCHED_PROCESS_IDS:
    scored = _scored_candidates(process_id, third_source_wer_by_task, "third_source")
    if not scored:
        continue

    ranked = sorted(scored, key=lambda r: r["combined_wer"])
    best = ranked[0]
    second_wer = ranked[1]["combined_wer"] if len(ranked) > 1 else np.nan

    third_source_match_rows.append({
        "process_ID": process_id,
        "best_match_id": best["candidate_id"],
        "best_combined_wer": best["combined_wer"],
        "pft_wer": best.get("PFT"),
        "sft_wer": best.get("SFT"),
        "ctd_wer": best.get("CTD"),
        "second_best_wer": second_wer,
        "margin": second_wer - best["combined_wer"],
    })

third_source_match_results = pd.DataFrame(third_source_match_rows)

# same confidence + zero-WER sanity criteria used for CognoSpeak-new / old CognoSpeak above
third_source_match_results["confident"] = (
    (third_source_match_results["best_combined_wer"] <= CONFIDENT_WER)
    & (third_source_match_results["margin"] >= CONFIDENT_MARGIN)
)
third_source_match_results["any_task_wer_zero"] = (
    third_source_match_results[["pft_wer", "sft_wer", "ctd_wer"]].eq(0).any(axis=1)
)

print(f"{len(third_source_match_results)}/{len(UNMATCHED_PROCESS_IDS)} unmatched participants "
      f"have at least one candidate in the third source")
print(third_source_match_results["confident"].value_counts())
third_source_match_results.sort_values("best_combined_wer").head(10)

102/102 unmatched participants have at least one candidate in the third source
confident
True     99
False     3
Name: count, dtype: int64


,process_ID,best_match_id,best_combined_wer,pft_wer,sft_wer,ctd_wer,second_best_wer,margin,confident,any_task_wer_zero
0,PROCESS-2_rec__123,R_60817_240527_213432,0.0,0.0,0.0,0.0,0.781769,0.781769,True,True
3,PROCESS-2_rec__150,R_26990_231231_171530,0.0,0.0,0.0,0.0,0.836565,0.836565,True,True
6,PROCESS-2_rec__187,R_19654_230327_103142,0.0,0.0,0.0,0.0,0.863553,0.863553,True,True
5,PROCESS-2_rec__126,S_18111_240522_102703,0.0,0.0,0.0,0.0,0.817121,0.817121,True,True
7,PROCESS-2_rec__278,R_06413_240518_132317,0.0,0.0,0.0,0.0,0.792464,0.792464,True,True
10,PROCESS-2_rec__353,R_75246_240520_145507,0.0,0.0,0.0,0.0,0.855184,0.855184,True,True
9,PROCESS-2_rec__318,R_71788_240209_163249,0.0,0.0,0.0,0.0,0.838235,0.838235,True,True
8,PROCESS-2_rec__374,MND_R_64641_231116_131735,0.0,0.0,0.0,0.0,0.000000,0.000000,False,True
12,PROCESS-2_rec__186,R_42005_230913_004912,0.0,0.0,0.0,0.0,0.828329,0.828329,True,True
13,PROCESS-2_rec__082,R_71134_240405_103735,0.0,0.0,0.0,0.0,0.843929,0.843929,True,True


In [193]:
third_source_match_results

,process_ID,best_match_id,best_combined_wer,pft_wer,sft_wer,ctd_wer,second_best_wer,margin,confident,any_task_wer_zero
0,PROCESS-2_rec__123,R_60817_240527_213432,0.000000,0.000000,0.000000,0.000000,0.781769,0.781769,True,True
1,PROCESS-2_rec__262,R_47039_231128_083014,0.016785,0.034483,0.015873,0.000000,0.812220,0.795434,True,True
2,PROCESS-2_rec__232,R_79665_230630_143948,0.010101,0.000000,0.030303,0.000000,0.871481,0.861380,True,True
3,PROCESS-2_rec__150,R_26990_231231_171530,0.000000,0.000000,0.000000,0.000000,0.836565,0.836565,True,True
4,PROCESS-2_rec__172,S_62330_240427_204939,0.004630,0.000000,0.013889,0.000000,0.823129,0.818500,True,True
...,...,...,...,...,...,...,...,...,...,...
97,PROCESS-2_rec__234,S_35666_240424_175439,0.010256,0.000000,0.030769,0.000000,0.901535,0.891279,True,True
98,PROCESS-2_rec__132,R_16136_230719_092345,0.000000,0.000000,0.000000,0.000000,0.956815,0.956815,True,True
99,PROCESS-2_rec__181,R_15644_240606_100244,0.016667,0.000000,0.000000,0.050000,0.938721,0.922054,True,True
100,PROCESS-2_rec__153,R_54511_240923_150240,0.931481,1.000000,0.933333,0.861111,0.931481,0.000000,False,False


In [194]:
third_source_match_results.to_csv(
    "/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/processed/third_source_match_results.csv",
    index=False,
)

In [196]:

THIRD_SOURCE_BASE_DIR = "/mnt/xdrive/cchat/Shared/CognoSpeak_NEW/MANUAL_TRANSCRIPTS_PER_QUESTION"
THIRD_SOURCE_EXTRA_DIR = "/mnt/xdrive/cchat/Shared/CognoSpeak_NEW/text/per_qs"      # TODO: fill in (the 3 additional participants)

def find_dir_names(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.is_dir():
        return set()
    return {p.name for p in base_dir.iterdir() if p.is_dir()}

third_source_dir_names = find_dir_names(THIRD_SOURCE_BASE_DIR) | find_dir_names(THIRD_SOURCE_EXTRA_DIR)

print(f"{len(third_source_dir_names)} participant folders found across both directories")
print(third_source_dir_names)

604 participant folders found across both directories
{'R_04952_240728_175453', 'R_96543_230228_110041', 'S_58836_240604_154333', 'R_04663_230704_121404', 'R_33085_241213_104044', 'R_56264_240904_140553', 'R_80701_231117_095923', 'R_13456_231125_103033', 'R_26990_231231_171530', 'R_92046_250226_105444', 'R_80701_240520_145731', 'S_96316_230630_104208', 'R_88710_241022_134020', 'R_43296_241128_122937', 'R_10853_250318_123115', 'R_69218_240809_100734', 'R_80598_240608_141818', 'S_21212_240112_072802', 'R_16481_240417_195838', 'R_52347_240903_220336', 'R_27369_241023_113638', 'R_39198_240821_122328', 'S_07238_230706_142032', 'S_20740_240320_111403', 'R_51249_230630_121032', 'R_99668_230711_071652', 'R_25893_240731_111923', 'S_89473_230714_132858', 'R_95044_240917_143434', 'R_61178_230202_135752', 'R_91842_241107_214854', 'R_47225_241221_203608', 'R_57203_230421_201304', 'R_87277_240531_161712', 'R_15644_240606_100244', 'R_70344_241009_105415', 'S_21278_240615_123155', 'S_00427_230703_1319

In [197]:
dir_names_a = find_dir_names(THIRD_SOURCE_BASE_DIR)
dir_names_b = find_dir_names(THIRD_SOURCE_EXTRA_DIR)

print(f"A: {len(dir_names_a)}, B: {len(dir_names_b)}")
print(f"in A but not B: {len(dir_names_a - dir_names_b)}")
print(f"in B but not A: {len(dir_names_b - dir_names_a)}")
print("only in B:", dir_names_b - dir_names_a)

A: 601, B: 604
in A but not B: 0
in B but not A: 3
only in B: {'R_57571_240523_123636', 'S_70454_230519_124655', 'S_13201_240321_092731'}


try to find missing two ids

In [198]:
missing_ids = ["PROCESS-2_rec__115", "PROCESS-2_rec__153"]

# meta_cogno so far is CognoSpeak-new only (see "Start with CognoSpeak new
# only..." cell above) -- bring old CognoSpeak's demographics into the pool
# too, since these two are exactly the ones without a transcript match yet
meta_cogno_old_demo = meta_CognoOld.copy()
meta_cogno_old_demo["gender_norm"] = meta_cogno_old_demo["gender"].apply(normalize_gender)
meta_cogno_old_demo["diagnosis_norm"] = meta_cogno_old_demo["diagnosis"]

meta_cogno_all = pd.concat([meta_cogno, meta_cogno_old_demo], ignore_index=True)

for process_id in missing_ids:
    prow = meta_process[meta_process["IDs"] == process_id].iloc[0]
    cands = get_candidates(prow, meta_cogno_all)
    print(f"{process_id}: gender={prow['gender_norm']}, age={prow['age']}, "
          f"diagnosis={prow['diagnosis_norm']}, MMSE={prow['MMSE']}")
    print(f"  -> {len(cands)} demographic candidate(s)")
    print(cands[["dir_name", "diagnosis", "age", "gender", "MMSE"]])
    print()

PROCESS-2_rec__115: gender=male, age=73, diagnosis=MCI, MMSE=27.0
  -> 4 demographic candidate(s)
                   dir_name diagnosis   age gender  MMSE
270   R_03155_250403_165147       MCI  73.0   male  27.0
852   R_55575_250610_140338       MCI  73.0   male  27.0
853   R_55575_251117_135042       MCI  73.0   male  27.0
1242  R_92154_251204_104156       MCI  73.0   male  27.0

PROCESS-2_rec__153: gender=female, age=81, diagnosis=HC, MMSE=nan
  -> 6 demographic candidate(s)
                   dir_name diagnosis   age  gender  MMSE
605   R_31792_250819_121256        HC  81.0  female   NaN
995   R_69404_240101_113544        HC  81.0  female   NaN
1150  R_84476_260318_110620        HC  81.0  female   NaN
2097  S_16476_240203_073313        HC  81.0  female   NaN
2551  S_47343_250205_120250        HC  81.0  female   NaN
2693  S_57876_231116_160729        HC  81.0  female   NaN

